In [1]:
# I'm downloading questionnaires in the form of PDF from afrobarometer and storing them in a folder

# I need to install libraries. 
from bs4 import BeautifulSoup       # makes scraping possible
import requests 
import os



In [5]:
# We need to have the webpage from where we are scraping.
url = "https://www.afrobarometer.org/surveys-and-methods/questionnaire/?select-countries%5B%5D=senegal&hidden-current-page=1"

# We need the folder to where we will store the data
output_folder = "/Users/albertolsen/Documents/Stud.Polit./11. Bologna/Development Economics/Questionnaires"

response = requests.get(url)
# print(response.text)

site = BeautifulSoup(response.text, 'html.parser')  # we've saved the parsed website as site


In [21]:
# We make a place holder variable
questionnaire_links = []

# We search for special key words inside the href find them and append them to the placeholder
for a in site.find_all('a', href=True):
    if "round" in a['href'].lower():
        #print(a['href'])
        questionnaire_links.append(a['href'])

print(questionnaire_links )



['https://www.afrobarometer.org/survey-resource/senegal-round-10-questionnaire-2025/', 'https://www.afrobarometer.org/survey-resource/senegal-round-9-questionnaire-2022/', 'https://www.afrobarometer.org/survey-resource/senegal-round-8-questionnaire/', 'https://www.afrobarometer.org/survey-resource/senegal-round-7-questionnaire/', 'https://www.afrobarometer.org/survey-resource/senegal-round-6-questionnaire/', 'https://www.afrobarometer.org/survey-resource/senegal-round-5-questionnaire/', 'https://www.afrobarometer.org/survey-resource/senegal-round-4-questionnaire/', 'https://www.afrobarometer.org/survey-resource/senegal-round-2-questionnaire-2002/', 'https://www.afrobarometer.org/survey-resource/senegal-round-3-questionnaire/']


In [31]:
# now we have to enter all the links and extract the pdf
# we do this by looping through all the links and inside there extract the pdf
pdf_links = []

for link in questionnaire_links:
    page = requests.get(link)       # page are all the links we've put in the placeholder
    page_soup = BeautifulSoup(page.text, 'html.parser')  # so here it is pressing one link at the time and parsing it. 

    for a in page_soup.find_all('a', href=True):
                if a['href'].endswith('.pdf'):
                        pdf_link = a['href']
                        pdf_links.append(pdf_link)

print(pdf_links)        


['https://www.afrobarometer.org/wp-content/uploads/2025/12/SEN_R10_Questionnaire_8Fev25_final.pdf', 'https://www.afrobarometer.org/wp-content/uploads/2023/06/SEN_R9.questionnaire_23Mai22_FINAL.docx.pdf', 'https://www.afrobarometer.org/wp-content/uploads/2022/02/afrobarometer_questionnaire_sen_r8_fr_2021-06-21.pdf', 'https://www.afrobarometer.org/wp-content/uploads/2022/02/afrobarometer_questionnaire_sen_r7_fr_2017-11-17.pdf', 'https://www.afrobarometer.org/wp-content/uploads/2022/02/sen_r6_questionnaire.pdf', 'https://www.afrobarometer.org/wp-content/uploads/2022/02/sen_r5_questionnaire.pdf', 'https://www.afrobarometer.org/wp-content/uploads/2022/02/sen_r4_questionnaire.pdf', 'https://www.afrobarometer.org/wp-content/uploads/2022/11/sen-R2quest11nov02-final.pdf', 'https://www.afrobarometer.org/wp-content/uploads/2022/02/sen_r3_questionnaire.pdf']


In [35]:
# so now I've collected all the pdf links and we just need to download them and save them in the folder

# first we make sure that we look like a browser, so we are not blocked by the website
headers = {"User-Agent": "Mozilla/5.0"}

for url in pdf_links: 
    print("Downloading:", url)
    response = requests.get(url, headers=headers, stream = True)    # we need the headers to not look suspicios and the stream to download the file in chunks
    if response.status_code == 200:
        filename = url.split("/")[-1]  # we split with each / and then take the last bit. 
        file_path = os.path.join(output_folder, filename)  # we join the output folder and the filename to get the full path
        with open(file_path, "wb") as f:    # we open the file in write binary mode, because pdfs are binary files. 
            for chunk in response.iter_content(chunk_size=8192):    # so what it does here is to take the pdf file in 8192 chunks, and below it gets written. This is because of 'streaming' so it is already in chunks
                f.write(chunk)    # we write the file in chunks to avoid memory issues
                print("Saved:", filename)
    else:
        print("failed:",url)





Downloading: https://www.afrobarometer.org/wp-content/uploads/2025/12/SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Questionnaire_8Fev25_final.pdf
Saved: SEN_R10_Qu

Okay, nedenstående virker ikke helt, men det er heller ikke dumt. Jeg prøver lige på en anden måde, hvor jeg begrænser på, at der skal stå "questionnaire" i linket. 

In [17]:
# Now I can search the webpage using the tree that I've stored.  

#We are making a base link from which we can extend upon with the links in 'site'
# base_url = "https://www.afrobarometer.org/"
# But this we actually don't need because the links are written in their full extend. 

# Now I create a subsample of the whole HTML, by restricting to a certain division.
rows = site.find_all("div", class_= "c-listing__list")

# now I can look inside this subsample for the proper links by using a 'loop' maybe?
for row in rows: 
    link = row.find("a")
    print(link['href'])


#questionnaire_links = []


https://www.afrobarometer.org/survey-resource/senegal-round-10-questionnaire-2025/
